In [1]:
## pip install tensorflow==2.15.0

In [2]:
import tensorflow as tf

print("tf.__version__ =", tf.__version__)


tf.__version__ = 2.15.0


In [3]:
# 샘플 예제

In [4]:
import numpy as np

def sigmoid(x):   # 시그모이드 함수
    return 1. / (1 + np.exp(-x))   # 수식을 표현한 내용

In [5]:
def numerical_derivative(f, x):   # f: 손실값을 계산하는 함수 / x: 업데이트 하기 위한 변수
    delta_x = 1e-4   # 미분할 값의 크기
    grad = np.zeros_like(x)
    it = np.nditer(x, flags=['multi_index'], op_flags=['readwrite'])
    
    while not it.finished:
        idx = it.multi_index        
        tmp_val = x[idx]
        x[idx] = float(tmp_val) + delta_x
        fx1 = f(x) # f(x+delta_x)
        
        x[idx] = float(tmp_val) - delta_x 
        fx2 = f(x) # f(x-delta_x)
        grad[idx] = (fx1 - fx2) / (2*delta_x)
        
        x[idx] = tmp_val 
        it.iternext()

    return grad

In [6]:
# https://f-lab.kr/insight/understanding-cross-entropy-loss-function

In [7]:
# 이게 핵심

class LogicGate:
    def __init__(self, gate_name, xdata, tdata):
        self.name = gate_name   # 게이트 이름
        self.xdata = xdata.reshape(4,2)   # 학습 데이터
        self.tdata = tdata.reshape(4,1)   # 정답 데이터

        self.W = np.random.rand(self.xdata.shape[1], 1)   # 가중치 초기값 설정
        self.b = np.random.rand(1)   # 바이어스 초기값 설정
        self.learning_rate = 1e-2    # 학습율

    def loss_func(self):
        delta = 1e-7
        z = np.dot(self.xdata, self.W) + self.b
        y = sigmoid(z)   # 활성함수: 시그모이드 함수 = 이진 분류에서 사용함
        return -np.sum(self.tdata*np.log(y+delta) + (1-self.tdata)*np.log((1-y)+delta))   # 이진 분류에 대한 손실값 / 이진 분류일 때 이 수식을 사용함

    
# 학습하는 과정을 함수로 만들 거임
    def train(self):
        f = lambda x : self.loss_func()   # 손실값 계산하는 람다 함수
        print("Initial loss value = ", self.loss_func())   # 초기 손실값 계산해서 출력
        for step in range(8001):
            self.W -= self.learning_rate * numerical_derivative(f, self.W)   # 가중치 업데이터
            self.b -= self.learning_rate * numerical_derivative(f, self.b)   # 바이어스 업데이트
            if (step % 1000 == 0):
                print("step = ", step, "loss value = ", self.loss_func())    # 헉습중 손실값 출력


# 예측하는 거 만들 거임
    def predict(self, input_data):
        z = np.dot(input_data, self.W) + self.b   # 회귀식 계산
        y = sigmoid(z)   # 활성 함수 시그모이드 함수 실행: 0~1 사이의 확률값 출력
        if y > 0.5:
            result = 1   # 분류값 저장
        else:
            result = 0
        return y, result

    
# 정확도 계산하는 거 만들 거임
    def accuracy(self, test_xdata, test_tdata):
        matched_list = []                                               # 정답과 일치한 내용 저장
        not_matched_list = []                                           # 정답과 불일치한 내용 저장
        for index in range(len(test_xdata)):
            (real_val, logical_val) = self.predict(test_xdata[index])
            if logical_val == test_tdata[index]:
                matched_list.append(index)                              # 정답과 일치한 내용 추가
            else:
                not_matched_list.append(index)                          # 정답과 불일치한 내용 추가
        accuracy_val = len(matched_list) / len(test_xdata)              # 정확도 계산
        return accuracy_val

In [8]:
# ===========================================================================================================

In [9]:
xdata = np.array([ [0, 0], [0, 1], [1, 0], [1, 1] ])   # 학습(train) 데이터
tdata = np.array([0, 0, 0, 1])                         # 정답(target) 데이터
AND_obj = LogicGate("AND_GATE", xdata, tdata)          # 모델 객체 생성
AND_obj.train()                                        # 학습

Initial loss value =  2.9405970412589255
step =  0 loss value =  2.919583913764392
step =  1000 loss value =  1.002246521939937
step =  2000 loss value =  0.6581917649130415
step =  3000 loss value =  0.49025048273713934
step =  4000 loss value =  0.3895292427702337
step =  5000 loss value =  0.3223909356246727
step =  6000 loss value =  0.27452682941326
step =  7000 loss value =  0.238745218178635
step =  8000 loss value =  0.21102658522597806


In [10]:
test_xdata = np.array([ [0, 0], [0, 1], [1, 0], [1, 1] ])
for input_data in test_xdata:
    (sigmoid_val, logical_val) = AND_obj.predict(input_data)
    print(input_data, '=', logical_val)

print('----------------------------------------')
test_tdata = np.array([0, 0, 0, 1])
accuracy_ret = AND_obj.accuracy(test_xdata, test_tdata)
print('Accuracy =>', accuracy_ret)

[0 0] = 0
[0 1] = 0
[1 0] = 0
[1 1] = 1
----------------------------------------
Accuracy => 1.0


In [11]:
# ======########################################################################

In [12]:
# OR 확인할 거임 
xdata = np.array([ [0, 0], [0, 1], [1, 0], [1, 1] ])   # 학습(train) 데이터
tdata = np.array([0, 1, 1, 1])                         # 정답(target) 데이터
OR_obj = LogicGate("OR_GATE", xdata, tdata)            # 모델 객체 생성
OR_obj.train()                                         # 학습

Initial loss value =  2.08847228849522
step =  0 loss value =  2.0794889583110536
step =  1000 loss value =  0.7262922279221963
step =  2000 loss value =  0.43437312457369515
step =  3000 loss value =  0.3051085515498847
step =  4000 loss value =  0.23345662909175607
step =  5000 loss value =  0.18834825602915067
step =  6000 loss value =  0.1575048509120449
step =  7000 loss value =  0.13515701193676158
step =  8000 loss value =  0.11825565415366678


In [13]:
test_xdata = np.array([ [0, 0], [0, 1], [1, 0], [1, 1] ])
for input_data in test_xdata:
    (sigmoid_val, logical_val) = OR_obj.predict(input_data)
    print(input_data, '=', logical_val)

print('----------------------------------------')
test_tdata = np.array([0, 0, 0, 1])
accuracy_ret = OR_obj.accuracy(test_xdata, test_tdata)
print('Accuracy =>', accuracy_ret)

[0 0] = 0
[0 1] = 1
[1 0] = 1
[1 1] = 1
----------------------------------------
Accuracy => 0.5


In [14]:
# =================================================================================

In [15]:
# NAND 확인할 거임 
xdata = np.array([ [0, 0], [0, 1], [1, 0], [1, 1] ])   # 학습(train) 데이터
tdata = np.array([1, 0, 0, 0])                         # 정답(target) 데이터
NAND_obj = LogicGate("NAND_GATE", xdata, tdata)        # 모델 객체 생성
NAND_obj.train()                                       # 학습

Initial loss value =  5.358969044737647
step =  0 loss value =  5.267398256366322
step =  1000 loss value =  0.7785317797298538
step =  2000 loss value =  0.453960069450572
step =  3000 loss value =  0.31505291952198883
step =  4000 loss value =  0.23938283263486712
step =  5000 loss value =  0.19225113125318655
step =  6000 loss value =  0.16025675513743312
step =  7000 loss value =  0.13719576469793604
step =  8000 loss value =  0.1198236774130131


In [16]:
test_xdata = np.array([ [0, 0], [0, 1], [1, 0], [1, 1] ])
for input_data in test_xdata:
    (sigmoid_val, logical_val) = NAND_obj.predict(input_data)
    print(input_data, '=', logical_val)

print('----------------------------------------')
test_tdata = np.array([0, 0, 0, 1])
accuracy_ret = OR_obj.accuracy(test_xdata, test_tdata)
print('Accuracy =>', accuracy_ret)

[0 0] = 1
[0 1] = 0
[1 0] = 0
[1 1] = 0
----------------------------------------
Accuracy => 0.5


In [17]:
# ===========================

In [18]:
# XOR 확인
xdata = np.array([ [0, 0], [0, 1], [1, 0], [1, 1] ])   # 학습(train) 데이터
tdata = np.array([0, 1, 1, 0])                         # 정답(target) 데이터
XOR_obj = LogicGate("XOR_GATE", xdata, tdata)          # 모델 객체 생성
XOR_obj.train()                                        # 학습

Initial loss value =  3.314959809581082
step =  0 loss value =  3.301288319359008
step =  1000 loss value =  2.7735415058788213
step =  2000 loss value =  2.7726285168038984
step =  3000 loss value =  2.772589656916342
step =  4000 loss value =  2.7725879964135274
step =  5000 loss value =  2.772587925411808
step =  6000 loss value =  2.772587922375508
step =  7000 loss value =  2.7725879222456618
step =  8000 loss value =  2.7725879222401093


In [19]:
test_xdata = np.array([ [0, 0], [0, 1], [1, 0], [1, 1] ])
for input_data in test_xdata:
    (sigmoid_val, logical_val) = XOR_obj.predict(input_data)
    print(input_data, '=', logical_val)

print('----------------------------------------')
test_tdata = np.array([0, 1, 1, 0])
accuracy_ret = XOR_obj.accuracy(test_xdata, test_tdata)
print('Accuracy =>', accuracy_ret)

[0 0] = 0
[0 1] = 0
[1 0] = 0
[1 1] = 1
----------------------------------------
Accuracy => 0.25


In [20]:
# ===============================================

In [21]:
# AND
xdata = np.array([ [0, 0], [0, 1], [1, 0], [1, 1] ])   # 학습(train) 데이터
tdata = np.array([0, 0, 0, 1])                         # 정답(target) 데이터
AND_obj = LogicGate("AND_GATE", xdata, tdata)          # 모델 객체 생성
AND_obj.train()                                        # 학습

Initial loss value =  4.677093168759532
step =  0 loss value =  4.616447363652497
step =  1000 loss value =  0.9948669400476922
step =  2000 loss value =  0.6551674491899642
step =  3000 loss value =  0.4885565159764189
step =  4000 loss value =  0.38844404098542384
step =  5000 loss value =  0.32163785024278896
step =  6000 loss value =  0.2739748726549889
step =  7000 loss value =  0.23832409660757303
step =  8000 loss value =  0.210695196429774


In [22]:
# OR 확인할 거임 
xdata = np.array([ [0, 0], [0, 1], [1, 0], [1, 1] ])   # 학습(train) 데이터
tdata = np.array([0, 1, 1, 1])                         # 정답(target) 데이터
OR_obj = LogicGate("OR_GATE", xdata, tdata)            # 모델 객체 생성
OR_obj.train()                                         # 학습

Initial loss value =  2.0294028348754964
step =  0 loss value =  2.0190644766141195
step =  1000 loss value =  0.6998849625902739
step =  2000 loss value =  0.4239583790830363
step =  3000 loss value =  0.2997107702120277
step =  4000 loss value =  0.23020362753449544
step =  5000 loss value =  0.18619064449031342
step =  6000 loss value =  0.15597604597886405
step =  7000 loss value =  0.13402031409838427
step =  8000 loss value =  0.11737900273336631


In [23]:
# NAND 확인할 거임 
xdata = np.array([ [0, 0], [0, 1], [1, 0], [1, 1] ])   # 학습(train) 데이터
tdata = np.array([1, 0, 0, 0])                         # 정답(target) 데이터
NAND_obj = LogicGate("NAND_GATE", xdata, tdata)        # 모델 객체 생성
NAND_obj.train()                                       # 학습

Initial loss value =  3.371897385384734
step =  0 loss value =  3.3244876258192573
step =  1000 loss value =  0.7350374123488915
step =  2000 loss value =  0.43767306946260454
step =  3000 loss value =  0.30679385106581597
step =  4000 loss value =  0.23446529943088282
step =  5000 loss value =  0.18901460955002072
step =  6000 loss value =  0.15797577157923892
step =  7000 loss value =  0.135506502596022
step =  8000 loss value =  0.11852481711435203


In [24]:
# 학습이 끝났으니 연결시켜주는 작업

input_data = np.array([[0,0],[0,1],[1,0],[1,1]])

s1 = []             # NAND 출력 저장
s2 = []             # OR 출력 저장
new_input_data = [] # AND 입력 저장
final_output = []   # 최종 출력 저장

for index in range(len(input_data)):
    s1 = NAND_obj.predict(input_data[index]) # S1에 값이 Y값 0~1 과 RESULT [0,1]
    s2 = OR_obj.predict(input_data[index])   # //
    new_input_data.append(s1[-1])            # -1이기에 RESULT값을 의미함
    new_input_data.append(s2[-1])            # 

    # Y 값           RESULT 
    (sigmoid_val, logical_val) = AND_obj.predict(np.array(new_input_data))
    final_output.append(logical_val)
    new_input_data = []

# 4번 결과값을 반복하며 출력
for index in range(len(input_data)):
    print(input_data[index], " = ", final_output[index])

[0 0]  =  0
[0 1]  =  0
[1 0]  =  0
[1 1]  =  0


# 다중 퍼셉트론 구조

In [25]:
# 딥러닝으로 XOR 문제 해결

In [26]:
import numpy as np

def sigmoid(x):   # 시그모이드 함수
    return 1. / (1 + np.exp(-x))   # 수식을 표현한 내용

In [27]:
def numerical_derivative(f, x):   # f: 손실값을 계산하는 함수 / x: 업데이트 하기 위한 변수
    delta_x = 1e-4   # 미분할 값의 크기
    grad = np.zeros_like(x)
    it = np.nditer(x, flags=['multi_index'], op_flags=['readwrite'])
    
    while not it.finished:
        idx = it.multi_index        
        tmp_val = x[idx]
        x[idx] = float(tmp_val) + delta_x
        fx1 = f(x) # f(x+delta_x)
        
        x[idx] = float(tmp_val) - delta_x 
        fx2 = f(x) # f(x-delta_x)
        grad[idx] = (fx1 - fx2) / (2*delta_x)
        
        x[idx] = tmp_val 
        it.iternext()

    return grad

In [28]:
# 이제 로직 게이트만 잘 설계해보자

In [29]:
# 직접 이렇게 설계할 일을 없지만 이해를 돕기 위해 딥 러닝이 어떤 구조인지 공부하기 위해 만든느 것임

class LogicGate:
    def __init__(self, gate_name, xdata, tdata): # 생성자 : __init__ # 해당 클래스가 사용할 변수를 초기화할 목적으로 사용
        self.name = gate_name
        self.xdata = xdata.reshape(4,2)  # 입력층 데이터 // 입력층에 대한 설계
        self.tdata = tdata.reshape(4,1)  # 정답 데이터

        self.W2 = np.random.rand(2,6)    # 은닉층 가중치 // 은닉층에 대한 설계
        self.b2 = np.random.rand(6)      # 은닉층 바이어스

        self.W3 = np.random.rand(6,1)    # 출력증 가중치 // 출력층에 대한 설계
        self.b3 = np.random.rand(1)      # 출력층 바이어스

        self.learning_rate = 1e-2        # 학습률

    def loss_val(self):                  # 손실값 계산 
        delta = 1e-7
        z2 = np.dot(self.xdata, self.W2) + self.b2
        a2 = sigmoid(z2)
        z3 = np.dot(a2, self.W3) + self.b3
        y = a3 = sigmoid(z3)
        return -np.sum(self.tdata*np.log(y+delta)+(1-self.tdata)*np.log((1 - y)+delta)) # 최종적인 y값을 가지고 손실값 계산

# 손실함수
    def train(self):
        f = lambda x : self.loss_val() # 손실함수 정의하는 형태 
        print("Initial loss value = ", self.loss_val()) # 초기손실값 계산하는 형태
        for step in range(5001): # 2만번 학습! >> 학습을 하게 되면 각 층에 있는 가중치와 바이어스를 업데이트 시켜야한다.
            self.W2 -= self.learning_rate * numerical_derivative(f, self.W2) # 은닉층의 가중치와 바이어스를 업데이트 시킴
            self.b2 -= self.learning_rate * numerical_derivative(f, self.b2) # 은닉층의 가중치와 바이어스를 업데이트 시킴
            self.W3 -= self.learning_rate * numerical_derivative(f, self.W3) # 출력층의 가중치와 바이어스를 업데이트 시킴
            self.b3 -= self.learning_rate * numerical_derivative(f, self.b3) # 출력층의 가중치와 바이어스를 업데이트 시킴
            if (step % 1000 == 0):                                           # 중간 확인 과정
                print("step = ", step, "loss value = ", self.loss_val())

    def predict(self, input_data):
        self.xdata = input_data
        z2 = np.dot(self.xdata, self.W2) + self.b2 # 은닉층에서 연산
        a2 = sigmoid(z2)                           # 은닉층에서 연산
        z3 = np.dot(a2, self.W3) + self.b3         # 출력층에서 연산
        y = a3 = sigmoid(z3)                       # 출력층에서 연산
        if y > 0.5:
            result = 1
        else:
            result = 0

        return y, result

    # 정확도 계산하는 거 만들 거임
    def accuracy(self, test_xdata, test_tdata):
        matched_list = []                                               # 정답과 일치한 내용 저장
        not_matched_list = []                                           # 정답과 불일치한 내용 저장
        for index in range(len(test_xdata)):
            (real_val, logical_val) = self.predict(test_xdata[index])
            if logical_val == test_tdata[index]:
                matched_list.append(index)                              # 정답과 일치한 내용 추가
            else:
                not_matched_list.append(index)                          # 정답과 불일치한 내용 추가
        accuracy_val = len(matched_list) / len(test_xdata)              # 정확도 계산
        return accuracy_val

In [30]:
# and 게이트
# AND
xdata = np.array([ [0, 0], [0, 1], [1, 0], [1, 1] ])   # 학습(train) 데이터
tdata = np.array([0, 0, 0, 1])                         # 정답(target) 데이터
AND_obj = LogicGate("AND_GATE", xdata, tdata)          # 모델 객체 생성
AND_obj.train()                                        # 학습

Initial loss value =  5.4438535298221025
step =  0 loss value =  5.2559817097728
step =  1000 loss value =  1.8964330368091618
step =  2000 loss value =  1.0546759025007315
step =  3000 loss value =  0.5089247886886599
step =  4000 loss value =  0.27693698756259916
step =  5000 loss value =  0.17501342479025075


In [31]:
test_data = np.array([ [0, 0], [0, 1], [1, 0], [1, 1] ])
for data in test_data:
    sigmoid_val, logical_val = AND_obj.predict(data)
    print(data, '=', logical_val)

print('----------------------------------------')
test_tdata = np.array([0, 0, 0, 1])
accuracy_ret = AND_obj.accuracy(test_data, test_tdata)
print('Accuracy =>', accuracy_ret)

[0 0] = 0
[0 1] = 0
[1 0] = 0
[1 1] = 1
----------------------------------------
Accuracy => 1.0


In [32]:
# xor 게이트가 잘 동작이 되는지가 중요하니 확인해보자

In [33]:
# XOR 게이트
# XOR
xdata = np.array([ [0, 0], [0, 1], [1, 0], [1, 1] ])   # 학습(train) 데이터
tdata = np.array([0, 1, 1, 0])                         # 정답(target) 데이터
XOR_obj = LogicGate("XOR_GATE", xdata, tdata)          # 모델 객체 생성
XOR_obj.train()                                        # 학습

Initial loss value =  4.902893820037183
step =  0 loss value =  4.800082088482567
step =  1000 loss value =  2.746890272040825
step =  2000 loss value =  2.708293703252722
step =  3000 loss value =  2.6122936425305108
step =  4000 loss value =  2.4149517988745735
step =  5000 loss value =  2.1470022801916944


In [34]:
test_data = np.array([ [0, 0], [0, 1], [1, 0], [1, 1] ])
for data in test_data:
    sigmoid_val, logical_val = XOR_obj.predict(data)
    print(data, '=', logical_val)

print('----------------------------------------')
test_tdata = np.array([0, 1, 1, 0]) # 정답
accuracy_ret = XOR_obj.accuracy(test_data, test_tdata)
print('Accuracy =>', accuracy_ret)

# 가중치의 초기치에 따라서 값이 다르게 나올 수도 있다. 초기치가 잘 잡히면 잘 나오는 경우가 생기게 된다.
# 이것은 AND 게이트는 학습을 조금해도 되지만  XOR 은 학습을 조금해도 안되니
# 학습을 조금 더 늘려보자!

# 그렇담 학습을 몇번 하는게 좋을까???
# 현재 2만번은 잘 돌아간다
# NODE 개수를 늘여주고 줄여주며, 

[0 0] = 0
[0 1] = 1
[1 0] = 1
[1 1] = 1
----------------------------------------
Accuracy => 0.75


In [35]:
# 어제까지는 은닉층이 1개인 예제를 다루어 보았다.
# 이제 은닉층이 2개인 것을 만들어보자!!